# Lab 17-03 · Analog Modulation and Its Noise

Five parts, each built on `analogmod.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import jv
import analogmod as am
rng = np.random.default_rng(1703)

## Part 1 · SSB by phasing

*Predict:* the output spectrum for a 300 Hz + 2 kHz message on a 10 kHz carrier, and the unwanted sideband's level with a 1° error in the 90° path.

In [2]:
fs, n = 100e3, 100_000
t = np.arange(n) / fs
m = np.cos(2 * np.pi * 300 * t) + 0.5 * np.sin(2 * np.pi * 2000 * t)
u = am.ssb(m, 10e3, fs)
U = np.abs(np.fft.rfft(u)) / n * 2
f = np.fft.rfftfreq(n, 1 / fs)
for fk in (8e3, 9.7e3, 10.3e3, 12e3):
    print(f"{fk / 1e3:5.1f} kHz: amplitude {U[int(fk * n / fs)]:.4f}")
def tone_ssb(eps_deg):
    mm = np.cos(2 * np.pi * 1e3 * t)
    out = mm * np.cos(2 * np.pi * 10e3 * t) - am.hilbert(mm) * np.sin(2 * np.pi * 10e3 * t + math.radians(eps_deg))
    X = np.abs(np.fft.rfft(out))
    return 20 * math.log10(X[int(11e3 * n / fs)] / X[int(9e3 * n / fs)])
print(f"suppression with 1 deg error: {tone_ssb(1.0):.2f} dB (formula {10 * math.log10(1 / math.tan(math.radians(0.5)) ** 2):.2f} dB)")
assert abs(U[int(10.3e3 * n / fs)] - 0.5) < 1e-6 and abs(U[int(12e3 * n / fs)] - 0.25) < 1e-6 and U[int(9.7e3 * n / fs)] < 1e-9
assert abs(tone_ssb(1.0) - 41.18) < 0.01

  8.0 kHz: amplitude 0.0000
  9.7 kHz: amplitude 0.0000
 10.3 kHz: amplitude 0.5000
 12.0 kHz: amplitude 0.2500
suppression with 1 deg error: 41.18 dB (formula 41.18 dB)


## Part 2 · Bessel lines

*Predict:* carrier and first sidebands for β = 1, 2.405 and 5.

In [3]:
fs, n = 1.536e6, 1 << 16
k0 = 640                                  # tone on an exact FFT bin: 15 kHz
t = np.arange(n) / fs
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, beta in zip(axes, (1.0, 2.405, 5.0)):
    z = np.exp(1j * beta * np.sin(2 * np.pi * k0 * fs / n * t))
    Z = np.abs(np.fft.fft(z)) / n
    meas = [Z[(k0 * k) % n] for k in range(-8, 9)]
    print(f"beta {beta}: carrier {meas[8]:.4f} (J0 = {abs(jv(0, beta)):.4f}), first pair {meas[9]:.4f} (J1 = {abs(jv(1, beta)):.4f})")
    assert max(abs(a - abs(jv(k, beta))) for a, k in zip(meas, range(-8, 9))) < 1e-9
    ax.stem(range(-8, 9), meas); ax.set_title(f"beta = {beta}")
plt.show()

beta 1.0: carrier 0.7652 (J0 = 0.7652), first pair 0.4401 (J1 = 0.4401)
beta 2.405: carrier 0.0001 (J0 = 0.0001), first pair 0.5191 (J1 = 0.5191)
beta 5.0: carrier 0.1776 (J0 = 0.1776), first pair 0.3276 (J1 = 0.3276)


## Part 3 · Four receivers

*Predict:* DSB, SSB and AM ($k_a$ = 0.9) output SNR at γ = 30 dB for the lab's three-tone message.

In [4]:
for sch in ("dsb", "ssb", "am"):
    snr, pm = am.simulate_linear_snr(sch, 30.0, rng=rng)
    th = am.snr_theory_db(sch, 30.0, pm=pm, ka=0.9)
    print(f"{sch}: simulated {snr:.2f} dB, formula {th:.2f} dB (message power {pm:.4f})")
    assert abs(snr - th) < 0.3

dsb: simulated 29.97 dB, formula 30.00 dB (message power 0.1668)


ssb: simulated 30.01 dB, formula 30.00 dB (message power 0.1668)


am: simulated 20.78 dB, formula 20.76 dB (message power 0.1668)


## Part 4 · FM above and below threshold

*Predict:* the FM output SNR (tone, D = 5) from γ = 12 to 40 dB, and where the simulation leaves the formula.

In [5]:
gs = np.arange(12, 41, 2.0)
res = {}
for fac in (1.0, 1.5):
    res[fac] = [am.simulate_fm_snr(g, rng=rng, if_factor=fac, n=1 << 19) for g in gs]
th = [am.snr_theory_db("fm", g) for g in gs]
for g, a, b, c in zip(gs, res[1.0], res[1.5], th):
    print(f"gamma {g:4.0f} dB: Carson IF {a:6.2f}, 1.5 x Carson IF {b:6.2f}, formula {c:6.2f}")
plt.plot(gs, th, label="formula"); plt.plot(gs, res[1.0], "o", label="IF = Carson"); plt.plot(gs, res[1.5], "s", label="IF = 1.5 Carson")
plt.xlabel("gamma (dB)"); plt.ylabel("output SNR (dB)"); plt.legend(); plt.show()
i30 = list(gs).index(30.0)
assert abs(res[1.5][i30] - th[i30]) < 0.3 and res[1.0][0] < th[0] - 10

gamma   12 dB: Carson IF   6.18, 1.5 x Carson IF   2.37, formula  27.74
gamma   14 dB: Carson IF  10.76, 1.5 x Carson IF   6.64, formula  29.74
gamma   16 dB: Carson IF  16.73, 1.5 x Carson IF  11.20, formula  31.74
gamma   18 dB: Carson IF  25.31, 1.5 x Carson IF  17.55, formula  33.74
gamma   20 dB: Carson IF  34.09, 1.5 x Carson IF  26.82, formula  35.74
gamma   22 dB: Carson IF  37.70, 1.5 x Carson IF  36.05, formula  37.74
gamma   24 dB: Carson IF  39.55, 1.5 x Carson IF  39.58, formula  39.74
gamma   26 dB: Carson IF  41.62, 1.5 x Carson IF  41.71, formula  41.74
gamma   28 dB: Carson IF  43.45, 1.5 x Carson IF  43.65, formula  43.74
gamma   30 dB: Carson IF  45.54, 1.5 x Carson IF  45.78, formula  45.74
gamma   32 dB: Carson IF  47.30, 1.5 x Carson IF  47.91, formula  47.74
gamma   34 dB: Carson IF  48.97, 1.5 x Carson IF  49.73, formula  49.74
gamma   36 dB: Carson IF  50.50, 1.5 x Carson IF  51.80, formula  51.74
gamma   38 dB: Carson IF  52.08, 1.5 x Carson IF  53.74, formula

## Part 5 · FM's parabolic noise and de-emphasis

*Predict:* the shape of the discriminator's output noise spectrum, and how much 75 µs de-emphasis removes over 15 kHz.

In [6]:
fs, n, W = 480e3, 1 << 20, 15e3
z = np.ones(n, dtype=complex)                    # unmodulated carrier, |z| = 1
n0 = 1e-7
r = z + math.sqrt(n0 * fs) * (rng.standard_normal(n) + 1j * rng.standard_normal(n))
fi = am.fm_discriminator(r, fs)
F = np.abs(np.fft.rfft(fi)) ** 2 / (fs * n) * 2
fr = np.fft.rfftfreq(n, 1 / fs)
band = (fr > 1e3) & (fr < W)
coef = np.polyfit(fr[band] ** 2, F[band], 1)
pred = n0 / 1.0                                  # one-sided noise PSD = 2 f^2 N0 / A^2 with A = 1 (two-sided f^2 N0)
print(f"fitted slope of PSD vs f^2: {coef[0]:.3e}, predicted 2 N0 / A^2 = {2 * n0:.3e}")
fw = fr[fr <= W]
Fw = F[fr <= W]
f1 = 1 / (2 * math.pi * 75e-6)
gain = np.sum(Fw) / np.sum(Fw / (1 + (fw / f1) ** 2))
print(f"de-emphasis noise reduction over 15 kHz: {10 * math.log10(gain):.2f} dB (formula 13.20 dB)")
plt.plot(fr[fr < 60e3] / 1e3, F[fr < 60e3]); plt.xlabel("frequency (kHz)"); plt.ylabel("discriminator output PSD"); plt.show()
assert abs(coef[0] / (2 * n0) - 1) < 0.05 and abs(10 * math.log10(gain) - 13.20) < 0.3

fitted slope of PSD vs f^2: 2.062e-07, predicted 2 N0 / A^2 = 2.000e-07
de-emphasis noise reduction over 15 kHz: 13.17 dB (formula 13.20 dB)
